In [2]:
import os, time
import pandas as pd
from influxdb_client_3 import InfluxDBClient3, Point

token = os.environ.get("INFLUXDB_TOKEN")
org = "CS 218"
host = "https://us-east-1-1.aws.cloud2.influxdata.com"

client = InfluxDBClient3(host=host, database="article_test", token=token, org=org)

# Get News and score value
query = """
SELECT company, score, time
FROM "news_articles"
WHERE ("score" != 101)
ORDER BY time DESC
"""
score_table = client.query(query=query, language='sql')

df_score_table = score_table.to_pandas()

print(df_score_table)


     company  score                time
0     Google      8 2024-10-23 15:14:09
1      Apple      8 2024-10-23 15:13:16
2     Google      7 2024-10-23 15:13:00
3      Apple      7 2024-10-23 15:12:55
4     Google      8 2024-10-23 15:09:32
...      ...    ...                 ...
1368   Apple     -4 2024-09-20 10:57:06
1369    Meta     -4 2024-09-20 10:57:06
1370  Amazon     -4 2024-09-20 10:57:06
1371   Apple      3 2024-09-20 08:00:00
1372  Google      0 2024-09-20 08:00:00

[1373 rows x 3 columns]


In [3]:
df_score_table = df_score_table.sort_values(by=['company', 'time'], ascending=[True, True])
df_score_table = df_score_table.reset_index(drop=True)
print(df_score_table)

     company  score                time
0     Amazon     -4 2024-09-20 10:57:06
1     Amazon      2 2024-09-24 17:25:09
2     Amazon      7 2024-09-25 14:44:58
3     Amazon      7 2024-09-25 15:38:19
4     Amazon      2 2024-09-25 19:54:07
...      ...    ...                 ...
1368  Nvidia     -4 2024-10-23 14:19:00
1369  Nvidia     -7 2024-10-23 14:41:25
1370  Nvidia      8 2024-10-23 14:54:32
1371  Nvidia      6 2024-10-23 15:00:00
1372  Nvidia      8 2024-10-23 15:04:40

[1373 rows x 3 columns]


In [4]:
client = InfluxDBClient3(host=host, database="stock_value_daily", token=token, org=org)

# Get daily stock value
query = """
SELECT *
FROM "stock_data"
ORDER BY time DESC
"""
daily_table = client.query(query=query, language='sql')

df_daily_table = daily_table.to_pandas()

print(df_daily_table)
print('-'*100)

client = InfluxDBClient3(host=host, database="stock_value_hourly", token=token, org=org)

hourly_table = client.query(query=query, language='sql')

df_hourly_table = hourly_table.to_pandas()

print(df_hourly_table)
print('-'*100)


     close_price  high_price interval  low_price  open_price ticker  \
0         235.86    236.2200      day    232.600     233.885   AAPL   
1         189.70    191.5201      day    186.975     188.350   AMZN   
2         165.14    165.7700      day    162.980     162.980  GOOGL   
3         582.01    583.5300      day    572.120     574.290   META   
4         143.59    144.4200      day    141.780     142.910   NVDA   
..           ...         ...      ...        ...         ...    ...   
115       228.87    229.8200      day    224.630     224.990   AAPL   
116       189.87    190.9900      day    188.470     190.040   AMZN   
117       162.14    163.7900      day    161.340     163.710  GOOGL   
118       559.10    562.0700      day    546.520     550.000   META   
119       117.87    119.6600      day    117.250     117.350   NVDA   

                   time     volume      vwap  
0   2024-10-22 04:00:00   35412947  234.8426  
1   2024-10-22 04:00:00   29171135  189.6999  
2   20

In [5]:
df_stock_table = pd.concat([df_daily_table, df_hourly_table], ignore_index=True)
df_stock_table = df_stock_table.sort_values(by=['ticker', 'time'], ascending=[True, True])
df_stock_table = df_stock_table.reset_index(drop=True)


# Edit ticker to company name 
ticker_to_company = {
    'AAPL': 'Apple',
    'AMZN': 'Amazon',
    'GOOGL': 'Google',
    'META': 'Meta',
    'NVDA': 'Nvidia'
}

df_stock_table['ticker'] = df_stock_table['ticker'].replace(ticker_to_company)
df_stock_table = df_stock_table.rename(columns={'ticker': 'company'})

# Print the combined DataFrame
print(df_stock_table)


      close_price  high_price interval  low_price  open_price company  \
0          228.87      229.82      day     224.63      224.99   Apple   
1          224.50      224.76     hour     222.79      222.79   Apple   
2          224.38      224.49     hour     223.87      224.34   Apple   
3          224.84      224.85     hour     224.13      224.23   Apple   
4          224.70      224.89     hour     224.01      224.50   Apple   
...           ...         ...      ...        ...         ...     ...   
2035       143.57      143.86     hour     143.21      143.30  Nvidia   
2036       143.56      143.62     hour     143.12      143.58  Nvidia   
2037       143.34      143.59     hour     143.24      143.56  Nvidia   
2038       143.20      143.37     hour     143.10      143.33  Nvidia   
2039       143.33      143.47     hour     143.18      143.18  Nvidia   

                    time    volume      vwap  
0    2024-09-19 04:00:00  65015591  228.3850  
1    2024-09-19 08:00:00     

In [6]:
print(df_stock_table.head(20))
print('-'*100)
print(df_stock_table.tail(20))


    close_price  high_price interval  low_price  open_price company  \
0      228.8700    229.8200      day   224.6300    224.9900   Apple   
1      224.5000    224.7600     hour   222.7900    222.7900   Apple   
2      224.3800    224.4900     hour   223.8700    224.3400   Apple   
3      224.8400    224.8500     hour   224.1300    224.2300   Apple   
4      224.7000    224.8900     hour   224.0100    224.5000   Apple   
5      225.1050    225.2900     hour   223.0000    224.1400   Apple   
6      227.3000    227.4200     hour   224.6300    225.1000   Apple   
7      227.9400    229.8200     hour   227.2300    227.2900   Apple   
8      228.3700    228.8699     hour   227.6600    227.9600   Apple   
9      229.1800    229.3300     hour   228.2800    228.3700   Apple   
10     229.3800    229.4600     hour   228.7150    229.1800   Apple   
11     229.3200    229.4400     hour   228.4000    229.3800   Apple   
12     228.8100    229.5900     hour   228.2800    229.3200   Apple   
13    

In [7]:
print(df_stock_table.info())
print(df_score_table.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2040 entries, 0 to 2039
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   close_price  2040 non-null   float64       
 1   high_price   2040 non-null   float64       
 2   interval     2040 non-null   object        
 3   low_price    2040 non-null   float64       
 4   open_price   2040 non-null   float64       
 5   company      2040 non-null   object        
 6   time         2040 non-null   datetime64[ns]
 7   volume       2040 non-null   int64         
 8   vwap         2040 non-null   float64       
dtypes: datetime64[ns](1), float64(5), int64(1), object(2)
memory usage: 143.6+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1373 entries, 0 to 1372
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   company  1373 non-null   object        
 1   score    1373 non-nul

# Important note: Daily stock is the daily value on that timestamp day (even if the timestamp says 4am)


# Table by content (all panda dataframes)
- - df_stock_table: stock table sorted by ticker and time (all tickers are together)

     Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   close_price  2040 non-null   float64       
 1   high_price   2040 non-null   float64       
 2   interval     2040 non-null   object        
 3   low_price    2040 non-null   float64       
 4   open_price   2040 non-null   float64       
 5   company      2040 non-null   object        
 6   time         2040 non-null   datetime64 <br>
 7   volume       2040 non-null   int64         
 8   vwap         2040 non-null   float64       

- df_score_table: news article and score sorted by company and time (ascending) (all companies news follow one another in the table)

     Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   company  1373 non-null   object        
 1   score    1373 non-null   int64         
 2   time     1373 non-null   datetime64


# Machine learning content starts here

In [23]:
is_amazon = df_score_table['company'] == 'Amazon'
is_google = df_score_table['company'] == 'Google'
df_score_table['time'] = df_score_table['time'].dt.tz_localize('UTC')
df_score_table['time'] = pd.to_datetime(df_score_table['time'])
print(df_score_table[is_amazon])
print(df_score_table[is_google])


    company  score                      time
0    Amazon     -4 2024-09-20 10:57:06+00:00
1    Amazon      2 2024-09-24 17:25:09+00:00
2    Amazon      7 2024-09-25 14:44:58+00:00
3    Amazon      7 2024-09-25 15:38:19+00:00
4    Amazon      2 2024-09-25 19:54:07+00:00
..      ...    ...                       ...
187  Amazon      7 2024-10-23 14:54:14+00:00
188  Amazon      7 2024-10-23 14:58:00+00:00
189  Amazon      8 2024-10-23 15:00:00+00:00
190  Amazon      8 2024-10-23 15:03:26+00:00
191  Amazon     -7 2024-10-23 15:08:47+00:00

[192 rows x 3 columns]
    company  score                      time
485  Google      0 2024-09-20 08:00:00+00:00
486  Google      7 2024-09-20 11:30:33+00:00
487  Google      6 2024-09-20 12:30:37+00:00
488  Google     -6 2024-09-21 10:30:00+00:00
489  Google      8 2024-09-22 12:03:00+00:00
..      ...    ...                       ...
883  Google     -7 2024-10-23 15:08:26+00:00
884  Google     -7 2024-10-23 15:08:47+00:00
885  Google      8 2024-10-23 1

In [24]:
start_time = pd.Timestamp('2024-09-20 10:00:00', tz='UTC')
end_time = pd.Timestamp('2024-09-22 12:00:00', tz='UTC')
filtered_df = df_score_table[(df_score_table['time'] >= start_time) & (df_score_table['time'] <= end_time)]
print(filtered_df)


     company  score                      time
0     Amazon     -4 2024-09-20 10:57:06+00:00
193    Apple     -4 2024-09-20 10:57:06+00:00
194    Apple      7 2024-09-20 11:30:33+00:00
195    Apple      6 2024-09-20 12:30:37+00:00
196    Apple      5 2024-09-20 20:44:59+00:00
197    Apple     -4 2024-09-21 10:30:00+00:00
486   Google      7 2024-09-20 11:30:33+00:00
487   Google      6 2024-09-20 12:30:37+00:00
488   Google     -6 2024-09-21 10:30:00+00:00
888     Meta     -4 2024-09-20 10:57:06+00:00
889     Meta      7 2024-09-20 14:15:08+00:00
890     Meta      8 2024-09-20 15:50:53+00:00
1105  Nvidia      0 2024-09-20 11:11:34+00:00
1106  Nvidia      7 2024-09-20 17:39:36+00:00
1107  Nvidia     -6 2024-09-21 01:32:58+00:00
1108  Nvidia     -8 2024-09-22 10:08:02+00:00
